# AMC 2026 – Round 3 · Strategy B: country value and momentum
Matterhorn Capital Management · group 3

The same steps as the TA's notebook (`02_round3_macro_lecture.ipynb`, §3), with three changes:
1. **21 countries**: the TA's 18 plus China (MCHI), India (INDA) and South Africa (EZA);
2. **value = raw P/B and P/B adjusted for ROE**, to reduce the sector bias of P/B (see `Macro/reports/Country value signals sector bias.md`);
3. **no tuning**: the parameters are fixed in advance from the literature, so the whole sample 2002–2026 is a test, and we add tests of the 1-month prediction (rank IC, hit rate, rolling IC).

The data come from our Bloomberg template → `01_bloomberg_macro_data.ipynb` (`r7_countries.csv`, `r7_pb.csv`, `r7_roe.csv`, `r7_rf_long.csv`, `r7_live.csv`).
The position size is decided later, together with Strategy A (TA notebook 02, §5–6).

## 0. Settings

In [1]:
import numpy as np                     # arrays and matrix algebra
import pandas as pd                    # tables with dates
import matplotlib.pyplot as plt        # plots
pd.set_option("display.float_format", "{:.4f}".format)

# Fixed in advance, not tuned (Asness, Moskowitz & Pedersen 2013; TA notebook 02)
MOM_LONG, MOM_SKIP = 12, 1             # momentum: return from t-12 to t-1, skipping the last month
N_TOP    = 4                           # countries we buy (about the top 20% of 21), as the TA
STRATEGY = "V+M"                       # main version, declared before running: 50/50 value and momentum
ROE_MAX  = 50                          # ROE above 50% treated as a data error (e.g. Mexico 2006)
MIN_CTRY = 8                           # countries needed in a month to run the P/B-ROE regression
DISCOUNT = 0.5                         # Class 4: halve the backtested Sharpe Ratio
ROLL_IC  = 36                          # months of the rolling IC

COUNTRY = {"EWA": "Australia", "EWC": "Canada", "EWG": "Germany", "EWJ": "Japan", "EWU": "United Kingdom",
           "EWQ": "France", "EWI": "Italy", "EWP": "Spain", "EWL": "Switzerland", "EWN": "Netherlands",
           "EWD": "Sweden", "EWH": "Hong Kong", "EWS": "Singapore", "EWY": "South Korea", "EWT": "Taiwan",
           "EWZ": "Brazil", "EWW": "Mexico", "SPY": "United States", "MCHI": "China", "INDA": "India",
           "EZA": "South Africa"}

def stats(r, rf, bench):
    # the Class 2 / Class 4 measures, as in TA notebook 02
    ex, exb = r - rf, bench - rf
    T = len(r) / 12
    beta, a = np.polyfit(exb, ex, 1)
    eps = ex - a - beta * exb
    IR = a * 12 / (eps.std() * np.sqrt(12)) if eps.std() > 0 else np.nan
    if r.equals(bench):
        beta, a, IR = 1.0, 0.0, np.nan
    wealth = (1 + r).cumprod()
    return pd.Series({"mean": r.mean() * 12, "vol": r.std() * np.sqrt(12),
                      "Sharpe": ex.mean() / r.std() * np.sqrt(12), "t(mean)": ex.mean() / r.std() * np.sqrt(12 * T),
                      "max drawdown": (wealth / wealth.cummax() - 1).min(), "worst month": r.min(),
                      "skewness": r.skew(), "beta": beta, "alpha": a * 12, "t(alpha)": IR * np.sqrt(T)})

## 1. Data
Month-end values. The last month in the files is October 2026; we add one empty row for **November 2026**, the month whose end (Friday 27 November) is our decision date. Momentum for that row only needs prices up to October (the last month is skipped anyway); P/B and ROE are carried forward from October.

In [2]:
P   = pd.read_csv("r7_countries.csv", index_col=0, parse_dates=True)          # total return index of the ETFs (USD)
pb  = pd.read_csv("r7_pb.csv",  index_col=0, parse_dates=True).reindex(index=P.index, columns=P.columns)
roe = pd.read_csv("r7_roe.csv", index_col=0, parse_dates=True).reindex(index=P.index, columns=P.columns)
rf  = pd.read_csv("r7_rf_long.csv", index_col=0, parse_dates=True)["rf"]
live = pd.read_csv("r7_live.csv").set_index("ticker")

DECISION = P.index[-1] + pd.offsets.MonthEnd(1)                                # 30 Nov 2026 (we trade at the 27 Nov close)
P   = P.reindex(P.index.append(pd.DatetimeIndex([DECISION])))
pb, roe = pb.reindex(P.index).ffill(limit=1), roe.reindex(P.index).ffill(limit=1)
roe = roe.where(roe <= ROE_MAX)                                                  # drop the data errors
R_c = P.pct_change(fill_method=None)

first = P.apply(pd.Series.first_valid_index).dt.strftime("%Y-%m")
print(f"{P.shape[1]} countries, {P.index[0]:%b %Y} -> {P.index[-2]:%b %Y}; decision row {DECISION:%d %b %Y}")
print("Late starters:", first[first > first.min()].to_dict())

21 countries, Jan 2001 -> Oct 2026; decision row 30 Nov 2026
Late starters: {'MCHI': '2011-03', 'INDA': '2012-02', 'EZA': '2003-02'}


## 2. Signals
**Momentum** (as the TA): return from month $t-12$ to $t-1$,
$$MOM_{i,t} = \frac{P_{i,t-1}}{P_{i,t-12}} - 1 .$$

**Value**, the average of two percentile ranks:
- raw P/B, as the TA: cheap = low P/B;
- P/B adjusted for ROE: each month, across countries, regress $\ln(P/B)$ on ROE and keep the residual,
$$\ln(P/B)_{i,t} = a_t + b_t\,ROE_{i,t} + e_{i,t}.$$
A negative $e_{i,t}$ means the country is cheaper than its profitability justifies (Wilcox 1984, P/B–ROE model). This removes much of the sector effect: bank-heavy markets have low ROE and low P/B by construction, tech-heavy ones high ROE and high P/B. The regression uses only month-$t$ data, so there is no look-ahead. A country without ROE keeps the raw P/B rank only.

**V+M** = average of the value and momentum ranks (no P/B → momentum only), as the TA.

In [3]:
rk = lambda x: x.rank(axis=1, pct=True) * 100                     # percentile rank across countries, 100 = best

mom = P.shift(MOM_SKIP) / P.shift(MOM_LONG) - 1

def roe_residual(lpb, roe):
    # cross-sectional regression of ln(P/B) on ROE, one month at a time
    out = pd.DataFrame(np.nan, index=lpb.index, columns=lpb.columns)
    for t in lpb.index:
        y, x = lpb.loc[t], roe.loc[t]
        ok = y.notna() & x.notna()
        if ok.sum() >= MIN_CTRY:
            b, a = np.polyfit(x[ok], y[ok], 1)
            out.loc[t, ok] = y[ok] - (a + b * x[ok])
    return out

lpb   = np.log(pb.where(pb > 0))
resid = roe_residual(lpb, roe)

v_raw, v_roe = rk(-lpb), rk(-resid)                                   # cheap = high rank
value = ((v_raw + v_roe) / 2).fillna(v_raw)                           # no ROE -> raw P/B only

scores = {"Momentum": rk(mom), "Value": value}
scores["V+M"] = ((scores["Momentum"] + scores["Value"]) / 2).fillna(scores["Momentum"])   # no P/B -> momentum only

slope = pd.Series({t: np.polyfit(roe.loc[t][lpb.loc[t].notna() & roe.loc[t].notna()],
                                 lpb.loc[t][lpb.loc[t].notna() & roe.loc[t].notna()], 1)[0]
                   for t in lpb.index if (lpb.loc[t].notna() & roe.loc[t].notna()).sum() >= MIN_CTRY})
print(f"P/B-ROE slope b: median {slope.median():.3f} per ROE point; positive in {(slope > 0).mean():.0%} of months "
      "(higher ROE -> higher P/B, as the model expects)")
print(f"Correlation of the two value ranks: {v_raw.stack().corr(v_roe.stack()):.2f}")

P/B-ROE slope b: median 0.054 per ROE point; positive in 100% of months (higher ROE -> higher P/B, as the model expects)
Correlation of the two value ranks: 0.74


## 3. Backtest on the whole sample
Weights chosen at the end of month $t$, return earned in month $t+1$ (as the TA). Equal weights on the top 4. Benchmark: all available countries, equally weighted. The three versions are the only trials, declared before running.

In [4]:
def top_weights(score, n=N_TOP):
    # equal weights on the n countries with the highest score; rows with fewer than n signals stay NaN
    pick = score.rank(axis=1, ascending=False, method="first") <= n
    w = pick.astype(float).div(pick.sum(axis=1), axis=0)
    return w.where(score.notna().sum(axis=1) >= n)

W  = {k: top_weights(s) for k, s in scores.items()}
bt = pd.DataFrame({k: (w.shift(1) * R_c).sum(axis=1, min_count=1) for k, w in W.items()})
bt["All countries"] = R_c.mean(axis=1)
bt = bt.dropna()
rf_bt = rf.reindex(bt.index).ffill()
BENCH = "All countries"

res = bt.apply(lambda r: stats(r, rf_bt, bt[BENCH]))
print(f"BACKTEST {bt.index[0]:%b %Y} - {bt.index[-1]:%b %Y} ({len(bt)} months). Benchmark: all countries, equal weights")
display(res.style.format("{:.2f}"))

SR_bt = res.loc["Sharpe", STRATEGY]
print(f"{STRATEGY}: Sharpe {SR_bt:.2f} | realistic (x{DISCOUNT}) {DISCOUNT * SR_bt:.2f} | benchmark {res.loc['Sharpe', BENCH]:.2f}")
print("Trials: 3 (Momentum, Value, V+M), parameters fixed in advance, no tuning")

fig, ax = plt.subplots(figsize=(10, 4.2))
for k, c in {"V+M": "#2a78d6", "Momentum": "#eda100", "Value": "#eb6834", BENCH: "#6b6b6b"}.items():
    ax.plot((1 + bt[k]).cumprod(), color=c, lw=2.4 if k == STRATEGY else 1.3, label=k)
ax.set_yscale("log"); ax.set_title("Growth of USD 1 (log scale)")
ax.legend(frameon=False); plt.tight_layout(); plt.show()

BACKTEST Feb 2002 - Oct 2026 (297 months). Benchmark: all countries, equal weights


,Momentum,Value,V+M,All countries
mean,0.11,0.11,0.10,0.10
vol,0.20,0.20,0.20,0.18
Sharpe,0.45,0.47,0.44,0.45
t(mean),2.25,2.35,2.19,2.25
max drawdown,-0.64,-0.53,-0.63,-0.58
worst month,-0.27,-0.22,-0.24,-0.24
skewness,-0.52,-0.19,-0.56,-0.58
beta,1.01,1.03,1.00,1.00
alpha,0.01,0.01,0.00,0.00
t(alpha),0.44,0.70,0.20,nan


V+M: Sharpe 0.44 | realistic (x0.5) 0.22 | benchmark 0.45
Trials: 3 (Momentum, Value, V+M), parameters fixed in advance, no tuning


C:\Users\Utente\AppData\Local\Temp\ipykernel_5380\4049025863.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.legend(frameon=False); plt.tight_layout(); plt.show()


## 4. Does the score predict next month?
Three tests of the 1-month prediction:
1. **Rank IC**: each month, the rank correlation (Spearman) between the score at the end of month $t$ and the countries' returns in month $t+1$. Mean, t-stat and share of months with IC > 0. A mean IC around 0.05 is already useful.
2. **Top 4 vs all countries**: the monthly excess return of the portfolio over the benchmark, its t-stat and the share of months it wins.
3. **Rolling IC** over 36 months: does the prediction work in every period, or only in some (the literature finds country value weaker after 2005)?

In [5]:
def rank_ic(score, R, min_n=MIN_CTRY):
    # Spearman correlation across countries: score at t vs return in t+1
    s, r = score.shift(1).rank(axis=1), R.rank(axis=1)
    ok = s.notna() & r.notna()
    s, r = s.where(ok).rank(axis=1), r.where(ok).rank(axis=1)
    ic = s.corrwith(r, axis=1)
    return ic[ok.sum(axis=1) >= min_n].dropna()

IC = pd.DataFrame({k: rank_ic(s, R_c) for k, s in scores.items()}).loc[bt.index[0]:]
ex = bt[list(scores)].sub(bt[BENCH], axis=0)                           # monthly return over the benchmark

pred = pd.DataFrame({
    "mean IC":        IC.mean(),
    "t(IC)":          IC.mean() / IC.std() * np.sqrt(IC.count()),
    "IC > 0":         (IC > 0).mean(),
    "excess / year":  ex.mean() * 12,
    "t(excess)":      ex.mean() / ex.std() * np.sqrt(ex.count()),
    "beats bench":    (ex > 0).mean(),
    "months":         IC.count()})
display(pred.style.format({"mean IC": "{:.3f}", "t(IC)": "{:.2f}", "IC > 0": "{:.0%}", "excess / year": "{:.2%}",
                           "t(excess)": "{:.2f}", "beats bench": "{:.0%}", "months": "{:.0f}"}))

fig, ax = plt.subplots(figsize=(10, 3.8))
for k, c in {"V+M": "#2a78d6", "Momentum": "#eda100", "Value": "#eb6834"}.items():
    ax.plot(IC[k].rolling(ROLL_IC).mean(), color=c, lw=2.4 if k == STRATEGY else 1.3, label=k)
ax.axhline(0, color="#999", lw=1)
ax.set_title(f"Rolling {ROLL_IC}-month mean rank IC (score at t vs return in t+1)")
ax.legend(frameon=False); plt.tight_layout(); plt.show()

by_period = IC.groupby(np.where(IC.index.year <= 2013, "2002-2013", "2014-2026")).mean()
print("Mean IC by period (description only, nothing is chosen on it):"); display(by_period.style.format("{:.3f}"))

,mean IC,t(IC),IC > 0,excess / year,t(excess),beats bench,months
Momentum,-0.010,-0.56,49%,0.74%,0.48,51%,297
Value,0.006,0.33,53%,1.28%,0.89,51%,297
V+M,-0.004,-0.27,49%,0.29%,0.22,52%,297


Mean IC by period (description only, nothing is chosen on it):


C:\Users\Utente\AppData\Local\Temp\ipykernel_5380\223324036.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.legend(frameon=False); plt.tight_layout(); plt.show()


,Momentum,Value,V+M
2002-2013,0.009,0.009,0.011
2014-2026,-0.029,0.002,-0.018


## 5. Today: the countries to buy
The score of the decision row (end of November 2026, with momentum to October and P/B, ROE of October) → the top 4 of the main version, held from the 27 November close to 27 December.

In [6]:
t = DECISION
today = pd.DataFrame({"country": pd.Series(COUNTRY),
                      "momentum 12-1": mom.loc[t], "P/B": pb.loc[t], "ROE": roe.loc[t], "P/B-ROE residual": resid.loc[t],
                      "rank momentum": scores["Momentum"].loc[t], "rank value": scores["Value"].loc[t],
                      "score V+M": scores["V+M"].loc[t]}).sort_values("score V+M", ascending=False)
display(today.style.format({"momentum 12-1": "{:.1%}", "P/B": "{:.2f}", "ROE": "{:.1f}", "P/B-ROE residual": "{:+.2f}",
                            "rank momentum": "{:.0f}", "rank value": "{:.0f}", "score V+M": "{:.0f}"}))

buy = W[STRATEGY].loc[t].dropna()
buy = buy[buy > 0]
print(f"Strategy B ({STRATEGY}, top {N_TOP}) for 27 Nov - 27 Dec 2026, equal weights inside the sleeve:")
for k in buy.index:
    print(f"  {COUNTRY[k]:<15} {k:<5} {live.loc[k, 'name'] if k in live.index else '':<32} "
          f"price {live.loc[k, 'px_last'] if k in live.index else float('nan'):>8.2f}")

,country,momentum 12-1,P/B,ROE,P/B-ROE residual,rank momentum,rank value,score V+M
EWY,South Korea,72.1%,2.32,32.9,-0.71,100,74,87
EWJ,Japan,16.8%,1.98,10.7,-0.03,86,67,76
EWH,Hong Kong,6.2%,1.21,8.6,-0.44,33,93,63
EWA,Australia,17.9%,2.73,12.2,+0.24,90,33,62
EWW,Mexico,12.6%,2.32,17.2,-0.12,57,62,60
EWZ,Brazil,1.0%,1.63,17.6,-0.48,24,93,58
EWP,Spain,12.3%,2.16,15.4,-0.12,48,67,57
EWU,United Kingdom,14.6%,2.37,14.6,+0.01,71,43,57
EWC,Canada,16.6%,2.85,14.3,+0.20,81,31,56
EWS,Singapore,12.5%,2.14,11.9,+0.00,52,55,54


Strategy B (V+M, top 4) for 27 Nov - 27 Dec 2026, equal weights inside the sleeve:
  Australia       EWA   ISHARES MSCI AUSTRALIA ETF       price    27.37
  Japan           EWJ   ISHARES MSCI JAPAN ETF           price    87.25
  Hong Kong       EWH   ISHARES MSCI HONG KONG ETF       price    22.15
  South Korea     EWY   ISHARES MSCI SOUTH KOREA ETF     price   129.19


## 6. Invest
The position is the 4 ETFs printed above, equally weighted inside the macro sleeve. The size of the sleeve comes from the tracking-error budget, together with Strategy A: TA notebook 02, §5 (size) and §6 (shares and trades at the close of Friday 27 November).

For the macro page (Class 7): the **view** (cheap countries, adjusted for profitability, with positive momentum outperform over the next month), the **instruments** (iShares MSCI country ETFs), the **risk** (momentum crashes at market turning points; value has been weak since about 2005) and the **exit condition** (a country leaves the top 4 at the next monthly ranking).